In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Install Ultralytics (YOLO)
!pip install ultralytics opencv-python-headless

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00


This is the workhorse. Update the RAW_IMG_DIR and RAW_XML_DIR paths to point to where your current dataset lives in your Google Drive.

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET

# --- 1. CONFIGURATION PATHS ---
# Update BASE_DIR to where you extracted the author's dataset in Google Drive
BASE_DIR = "/content/drive/MyDrive/VOC_PCB"

ANN_DIR = os.path.join(BASE_DIR, "Annotations")
IMG_DIR = os.path.join(BASE_DIR, "JPEGImages") # Sometimes named 'JPEGImages' in the repo
SPLIT_DIR = os.path.join(BASE_DIR, "ImageSets/Main")  # Assuming train.txt, val.txt, test.txt are here

# YOLO Output Structure
YOLO_DIR = os.path.join(BASE_DIR, "yolo_ready")
splits = ['train', 'val', 'test']

for split in splits:
    os.makedirs(os.path.join(YOLO_DIR, f"images/{split}"), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DIR, f"labels/{split}"), exist_ok=True)

CLASSES = ["missing_hole", "mouse_bite", "open_circuit", "short", "spur", "spurious_copper"]

def process_split(split_name):
    txt_file = os.path.join(SPLIT_DIR, f"{split_name}.txt")
    if not os.path.exists(txt_file):
        print(f"Warning: {txt_file} not found. Skipping {split_name}.")
        return

    print(f"Processing {split_name} split...")

    with open(txt_file, 'r') as f:
        file_names = [line.strip() for line in f.readlines()]

    for file_name in file_names:
        if not file_name: continue

        xml_path = os.path.join(ANN_DIR, f"{file_name}.xml")
        img_path = os.path.join(IMG_DIR, f"{file_name}.jpg")

        # Output paths
        out_img_path = os.path.join(YOLO_DIR, f"images/{split_name}/{file_name}.jpg")
        out_lbl_path = os.path.join(YOLO_DIR, f"labels/{split_name}/{file_name}.txt")

        # 1. Check if source image and XML exist
        if not os.path.exists(xml_path) or not os.path.exists(img_path):
            continue

        # 2. Parse XML
        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find('size')
        img_w = float(size.find('width').text)
        img_h = float(size.find('height').text)

        yolo_labels = []
        for obj in root.findall('object'):
            cls_name = obj.find('name').text
            if cls_name not in CLASSES: continue
            cls_id = CLASSES.index(cls_name)

            bndbox = obj.find('bndbox')

            # 3. Apply Boundary Safety Clipping (Fixing the check_xml.py warnings)
            xmin = max(0.0, float(bndbox.find('xmin').text))
            ymin = max(0.0, float(bndbox.find('ymin').text))
            xmax = min(img_w, float(bndbox.find('xmax').text))
            ymax = min(img_h, float(bndbox.find('ymax').text))

            # Skip invalid boxes where min is greater than max after clipping
            if xmax <= xmin or ymax <= ymin:
                continue

            # 4. YOLO Normalization Math
            x_center = ((xmin + xmax) / 2) / img_w
            y_center = ((ymin + ymax) / 2) / img_h
            w = (xmax - xmin) / img_w
            h = (ymax - ymin) / img_h

            yolo_labels.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

        # 5. Write to YOLO Format and Copy Image
        if yolo_labels:
            with open(out_lbl_path, 'w') as f:
                f.write('\n'.join(yolo_labels))
            # Use shutil.copy2 to preserve metadata while copying to the YOLO directory
            shutil.copy2(img_path, out_img_path)

# Execute the pipeline
for split in splits:
    process_split(split)

print("Dataset transformation complete! It is fully ready for YOLO.")

Processing train split...
Processing val split...
Processing test split...
Dataset transformation complete! It is fully ready for YOLO.


YOLO needs a configuration file to know where your newly processed data is and what the classes mean. Run this cell to generate it automatically.

In [ ]:
import yaml

data_yaml = {
    'path': '/content/drive/MyDrive/VOC_PCB/yolo_ready',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',  # Newly added testing path
    'nc': 6,
    'names': ["missing_hole", "mouse_bite", "open_circuit", "short", "spur", "spurious_copper"]
}

with open('/content/drive/MyDrive/VOC_PCB/yolo_ready/pcb_data.yaml', 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)